# Análise Exploratória de Dados - Multivariada

O código apresentado neste notebook realiza uma análise exploratória entre as variáveis presentes na base de dados e tem como objetivo identificar padrões e verificar a viabilidade da base de dados para o contexto de justiça algorítimica. 

In [69]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats

In [70]:
df = pd.read_csv("../dataset/dataset_limpo_multivariado.csv")

### Dados sensíveis + Dados de proposta

Há alguns dados sensíveis principais que podem ser usados para uma investigação de justiça algorítmica. Na nossa base de dados eles são: 

- Idade
- Grau de instrução escolar
- Status civil
- Status econômico (tendo como proxy residência)
- Sexo biológico 

Abaixo vamos realizar uma exploração estatística entre as features/variaveis com o objetivo de verificar se possuímos correlações suficientes que justifiquem um aprofundamento em situações de (in)justiça algorítimica. 

#### **Dados de gênero**

Para facilitar analises futuras, vou criar uma coluna que indica a porcentagem de mulheres dentro daquele grupo de clientes

In [71]:
df['PORCENTAGEM_MULHERES_GRUPO'] = (df['QT_CLI_MLH']/df['QT_CLI']).round(1)

Acredito que possa ser mais interessante realizar uma separação do meu dataset em três grandes grupos demográficos: (1) o grupo em que devedor seja composto apenas por grupo de mulheres, (2) o grupo devedor seja composto apenas por grupo de homens e (3) o grupo devedor é um grupo misto em gênero.

In [72]:
df_mulheres = df[df['PORCENTAGEM_MULHERES_GRUPO'] == 1]
df_homens = df[df['PORCENTAGEM_MULHERES_GRUPO']  == 0]
df_misto = df[(0 < df['PORCENTAGEM_MULHERES_GRUPO']) & (df['PORCENTAGEM_MULHERES_GRUPO'] < 1)]

print(f'Há {len(df_mulheres)} linhas/registros de grupo de clientes composto apenas por mulheres')
print(f'Há {len(df_homens)} linhas/registros de grupos de clientes composto apenas por homens')
print(f'Há {len(df_misto)} linhas/registros de grupo misto em gênero composto')



Há 41571 linhas/registros de grupo de clientes composto apenas por mulheres
Há 7231 linhas/registros de grupos de clientes composto apenas por homens
Há 77857 linhas/registros de grupo misto em gênero composto


Observamos que em maioria os grupos de pessoas devedoras é homogêneo (77.857 grupos), seguido por grupos exclusivo de mulheres (41.571 grupos). É evidente portanto que há um desbalancemanto na base de dados, vou investigar melhor o impacto que esse desbalancemanto pode ter nas demais variáveis.

In [73]:
media_mulheres = df_mulheres['QT_CLI'].sum()/len(df_mulheres)
media_homens = df_homens['QT_CLI'].sum()/len(df_homens)
media_misto = df_misto['QT_CLI'].sum()/len(df_misto)

print(f'Há em média {media_mulheres:.0f} mulheres por contrato, no grupo de clientes exclusivo de mulheres')
print(f'Há em média {media_homens:.0f} homens por contrato, no grupo de clientes exclusivo de homens')
print(f'Há em média {media_misto:.0f} pessoas por contrato, no grupo de clientes misto em gênero')

Há em média 4 mulheres por contrato, no grupo de clientes exclusivo de mulheres
Há em média 4 homens por contrato, no grupo de clientes exclusivo de homens
Há em média 5 pessoas por contrato, no grupo de clientes misto em gênero


Arredondando as casas decimais (afinal não existe meia pessoa), observamos que em média a quantidade de pessoas por contrato/empréstimo em cada um dos grupos é a mesma.

- Análise sobre valores solicitados em relação à genero

In [74]:
subsets = {
    "Mulheres": df_mulheres,
    "Homens": df_homens,
    "Mistos": df_misto,
}

In [75]:
estatisticas = []

for nome, sub_df in subsets.items():
    s = sub_df["VR_SOL_PRP"]
    estatisticas.append(
        {
            "Subgrupo": nome,
            "Contratos": len(s),
            "Média": s.mean(),
            "Mediana": s.median(),
            "Máximo": s.max(),
        }
    )

df_valores_proposto_por_grupo = pd.DataFrame(estatisticas)
df_valores_proposto_por_grupo

,Subgrupo,Contratos,Média,Mediana,Máximo
0,Mulheres,41571,11873.459161,8662.0,840770.0
1,Homens,7231,13458.812619,9411.0,85124.0
2,Mistos,77857,13547.314878,9573.0,192810.0


A mediana solicitada pelas mulheres (R$ 8.662,00) é menor do que a dos homens (R$ 9.411,00) e dos clientes mistos em gênero (R$ 9.573,00). A média segue a mesma tendência: mulheres pedem em média R$ 11.873, enquanto homens e mistos pedem mais de R$ 13.400.

TODO: VERIFICAR SE ESSA DIFERENÇA PODE TER ALGUMA RELAÇÃO COM A NATUREZA DE NEGÓCIO. 

In [76]:
estatistica_per_capita = []

for nome, sub_df in subsets.items():
    sub_df["VR_SOL_PRP_PER_CAPITA"] = sub_df["VR_SOL_PRP"] / sub_df["QT_CLI"]

    estatistica_per_capita.append(
        {
            "Subgrupo": nome,
            "Contratos": len(sub_df["VR_SOL_PRP_PER_CAPITA"]),
            "Média Per Capita": sub_df["VR_SOL_PRP_PER_CAPITA"].mean(),
            "Mediana Per Capita": sub_df["VR_SOL_PRP_PER_CAPITA"].median(),
        }
    )

df_estat_per_capita = pd.DataFrame(estatistica_per_capita)
df_estat_per_capita

/tmp/ipykernel_52753/1763304335.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sub_df["VR_SOL_PRP_PER_CAPITA"] = sub_df["VR_SOL_PRP"] / sub_df["QT_CLI"]
/tmp/ipykernel_52753/1763304335.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sub_df["VR_SOL_PRP_PER_CAPITA"] = sub_df["VR_SOL_PRP"] / sub_df["QT_CLI"]
/tmp/ipykernel_52753/1763304335.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at o

,Subgrupo,Contratos,Média Per Capita,Mediana Per Capita
0,Mulheres,41571,2873.672055,2128.142857
1,Homens,7231,3577.775397,2625.000000
2,Mistos,77857,2972.949489,2108.000000


No valor bruto da proposta, os grupos mistos pareciam pedir mais dinheiro, mas ao ajustar o valor pela quantidade de clientes no contrato, a mediana por integrante de mulheres (R$ 2.128,14) e mistos (R$ 2.108,00) é praticamente idêntica, com uma diferença de apenas R$ 20,14. Entretanto, podemos ver que há uma demanda individual superior nos homens: O grupo exclusivo de homens pede tickets per capita consistentemente maiores com uma mediana de R$ 2.625,00 (cerca de 23% superior aos demais grupos).

In [81]:
valor_solicitado_per_capita = df["VR_SOL_PRP"] / df["QT_CLI"]

correlacao = df['PORCENTAGEM_MULHERES_GRUPO'].corr(valor_solicitado_per_capita, method="spearman")
print(f"A correlação entre as duas variáveis é: {correlacao:.2f}")

A correlação entre as duas variáveis é: -0.04


Por meio do teste de correlaçao de Spearman, verificamos que as duas variáveis são independentes de forma que o gênero do grupo não indica sua necessidade ou ambição. Porém, podemos verificar se essa ausência de correlação pode ser devido a outros fatores como tempo de relacionamento com o banco, dados de inadimplência, tipo de natureza do empreendimento e etc.

- Análise da relação entre o valor aprovado de proposta e o solicitado em relação à gênero

In [ ]:
df_copy = df.copy()

In [ ]:
df_copy['PRT_VLR_APR_SOL'] = df_copy['VR_APR_PRP']/df_copy['VR_SOL_PRP'] 

In [ ]:
df_mulheres = df_copy[df_copy['QT_CLI_MLH'] == df_copy['QT_CLI']]
df_homens = df_copy[df_copy['QT_CLI_HOM'] == df_copy['QT_CLI']]
df_misto = df_copy[(df_copy['QT_CLI_MLH'] != df_copy['QT_CLI']) & (df_copy['QT_CLI_MLH'] > 0)]

In [ ]:
subsets = {
    "Mulheres": df_mulheres,
    "Homens": df_homens,
    "Mistos": df_misto,
}

In [ ]:
estatisticas_valor_aprovado = []

for nome, sub_df in subsets.items():
    estatisticas_valor_aprovado.append(
        {
            "Subgrupo": nome,
            "Contratos": len(sub_df["PRT_VLR_APR_SOL"]),
            "Média": sub_df["PRT_VLR_APR_SOL"].mean(),
            "Mediana": sub_df["PRT_VLR_APR_SOL"].median(),
            "Máximo": sub_df["PRT_VLR_APR_SOL"].max(),
        }
    )

df_valores_aprovado_por_grupo = pd.DataFrame(estatisticas_valor_aprovado)
df_valores_aprovado_por_grupo

,Subgrupo,Contratos,Média,Mediana,Máximo
0,Mulheres,41571,0.922550,0.997529,2.652174
1,Homens,7231,0.934464,1.000000,3.135170
2,Mistos,77857,0.921289,0.986500,3.142857


In [ ]:
correlacao = df['PORCENTAGEM_MULHERES_GRUPO'].corr(df['PRT_VLR_APR_SOL'], method='spearman')
print(f"A correlação entre as duas variáveis é: {correlacao:.2f}")

A correlação entre as duas variáveis é: -0.02


Com a ajuda da correlação de Spearman (-0.02) vemos que não existe uma tendência de crescimento ou queda acumulada - buscava entender se a medida que a porcentagem de mulheres aumenta, a taxa de aprovação também aumenta/diminui, mesmo que de forma exponencial, lenta ou irregular. 

#### **Dados de Idade** 

,VR_IDD_MNM,VR_IDD_MAX,VR_IDD_MED
count,126659.000000,126659.000000,126659.000000
mean,28.962458,54.617651,40.530108
std,7.550142,11.469753,7.694171
min,-3.000000,20.000000,19.000000
25%,23.000000,47.000000,35.000000
50%,28.000000,55.000000,40.000000
75%,33.000000,63.000000,46.000000
max,72.000000,101.000000,77.000000
